# BSDT Sonar — H7: Advanced Annealing Schedules

**Standalone notebook** — run from top to bottom on Colab (GPU required).

Tests 9 annealing schedules at n = 500, 750, 1000 to find which maximises solve rate by maximising broadcast time (time at μ ≈ 0).

Author: Odeyemi Olusegun Israel, Independent Researcher, Derby UK

In [ ]:
import torch
import numpy as np
import time
import matplotlib.pyplot as plt

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# CORE ENGINE — BSDTSonarEngine
# ═══════════════════════════════════════════════════════════════════

class BSDTSonarEngine:
    """
    BSDT Sonar core engine. Consolidated from v22 with all breakthroughs.

    Energy landscape:
        E(s) = E_clause(s) + mu * sum_i (1 - s_i^2)^2

    where E_clause = sum_c prod_{i in c} (1 - sign_i * s_i) / 2

    Gradient flow with adaptive friction:
        ds/dt = -(1 + gamma) * grad E + noise
        gamma = E_clause / (E_clause + theta)
    """

    def __init__(self, n, num_instances=100, num_particles=2000,
                 alpha=3.0, mu_scale=0.1, device=device):
        self.n    = n
        self.ni   = num_instances
        self.np_  = num_particles
        self.alpha = alpha
        self.mu_scale = mu_scale
        self.device = device
        self.m    = int(alpha * n)

    # ── Instance generation ──────────────────────────────────────
    def generate_instances(self):
        ni, m, n = self.ni, self.m, self.n
        cv = torch.zeros(ni, m, 3, dtype=torch.long, device=self.device)
        cs = torch.zeros(ni, m, 3, dtype=torch.float, device=self.device)
        for inst in range(ni):
            for c in range(m):
                perm = torch.randperm(n, device=self.device)[:3]
                cv[inst, c] = perm
                cs[inst, c] = torch.randint(0, 2, (3,), device=self.device).float() * 2 - 1
        return cv, cs

    # ── Compute stabilisation parameter ──────────────────────────
    def compute_mu(self, clause_vars):
        ni, n = self.ni, self.n
        degrees = torch.zeros(ni, n, device=self.device)
        for pos in range(3):
            idx = clause_vars[:, :, pos]
            degrees.scatter_add_(1, idx, torch.ones_like(idx, dtype=torch.float))
        max_deg = degrees.max(dim=1).values
        lam_max = 0.25 * max_deg
        return (self.mu_scale * lam_max).clamp(min=0.01), lam_max

    # ── Energy + gradient (fused) ────────────────────────────────
    def energy_and_grad(self, s, cv, cs, mu):
        ni, np_, m, n = self.ni, s.shape[1], self.m, self.n

        cv4  = cv.unsqueeze(1).expand(ni, np_, m, 3)
        sexp = s.unsqueeze(2).expand(ni, np_, m, n)
        s_at = torch.gather(sexp, 3, cv4)
        cs4  = cs.unsqueeze(1).expand(ni, np_, m, 3)

        lit = (1.0 - cs4 * s_at) / 2.0
        l0, l1, l2 = lit[..., 0], lit[..., 1], lit[..., 2]

        E_clause = (l0 * l1 * l2).sum(dim=2)
        mu3 = mu.view(ni, 1, 1)
        E_stab = (mu3 * (1.0 - s**2)**2).sum(dim=2)
        E_total = E_clause + E_stab

        # Clause gradient
        dl0 = (-cs4[..., 0] / 2.0) * l1 * l2
        dl1 = l0 * (-cs4[..., 1] / 2.0) * l2
        dl2 = l0 * l1 * (-cs4[..., 2] / 2.0)

        g = torch.zeros(ni, np_, n, device=self.device)
        for pos, dl in enumerate([dl0, dl1, dl2]):
            idx = cv[:, :, pos].unsqueeze(1).expand(ni, np_, m)
            g.scatter_add_(2, idx, dl)

        # Stabilisation gradient
        g = g + mu3 * (-4.0 * s * (1.0 - s**2))

        return E_total, E_clause, g

    # ── Gradient flow with momentum + plateau escape ─────────────
    def gradient_flow(self, s, cv, cs, mu, steps, dt=0.05,
                      beta=0.90, plateau_window=50,
                      noise_boost=4.0, dt_boost=2.0,
                      FR_var=None, sparse_thr=0.1,
                      mu_override=None):
        ni, np_, n = self.ni, s.shape[1], self.n
        v = torch.zeros_like(s)
        plat_count = torch.zeros(ni, np_, device=self.device)
        best_E = torch.full((ni, np_), float('inf'), device=self.device)

        for step in range(steps):
            if mu_override is not None:
                mu_eff = mu_override(step, steps, mu)
            else:
                mu_eff = mu

            _, E_clause, g = self.energy_and_grad(s, cv, cs, mu_eff)

            improved = E_clause < best_E
            best_E = torch.where(improved, E_clause, best_E)
            plat_count = torch.where(improved, torch.zeros_like(plat_count),
                                     plat_count + 1)

            plat_mask = plat_count >= plateau_window
            plat_frac = plat_mask.float().mean().item()

            decay  = 1.0 / (1.0 + 0.002 * step)
            gnorm  = g.norm(dim=2, keepdim=True).clamp(min=1e-10)
            dt_eff = dt * decay / (1.0 + 0.05 * gnorm)

            dt_eff = dt_eff * torch.where(
                plat_mask.unsqueeze(2),
                torch.full_like(dt_eff, dt_boost),
                torch.ones_like(dt_eff)
            )

            E_c = E_clause.clamp(min=0)
            gamma = (E_c / (E_c + 1.0)).unsqueeze(2)

            v = beta * v - dt_eff * (1.0 + gamma) * g

            base_n = 0.03 * decay
            ns = torch.where(
                plat_mask.unsqueeze(2),
                torch.full_like(v, base_n * noise_boost),
                torch.full_like(v, base_n)
            )
            noise = torch.randn_like(s) * ns
            s_new = torch.clamp(s + v + noise, -1.0, 1.0)

            if FR_var is not None and plat_frac > 0.3:
                weak = (FR_var < sparse_thr).unsqueeze(1)
                reset = plat_mask.unsqueeze(2) & weak
                rand_v = torch.empty(ni, np_, n, device=self.device).uniform_(-0.5, 0.5)
                s_new = torch.where(reset, rand_v, s_new)
                v = torch.where(reset, torch.zeros_like(v), v)
                plat_count = torch.where(plat_mask, torch.zeros_like(plat_count), plat_count)

            s = s_new

        return s, best_E

    # ── Evaluate solve rate ──────────────────────────────────────
    def evaluate(self, s, cv, cs, mu):
        s_round = torch.sign(s + 1e-10)
        _, E, _ = self.energy_and_grad(s_round, cv, cs, mu)
        best_per = E.clamp(min=0).min(dim=1).values
        solved = (best_per < 0.5).float().mean().item()
        se = np.sqrt(solved * (1 - solved) / max(self.ni, 1))
        return solved, se, best_per.float().mean().item()

print('BSDTSonarEngine loaded.')

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# H7: ADVANCED ANNEALING SCHEDULES
# ═══════════════════════════════════════════════════════════════════

# ── Schedule definitions ─────────────────────────────────────────

def sched_linear(t):
    """Linear: 50% time below μ/2"""
    return t

def sched_cosine(t):
    """Cosine: 75% time below μ/2"""
    return (1.0 - np.cos(np.pi * t)) / 2.0

def sched_cosine2(t):
    """Cosine²: ~85% time below μ/2"""
    return ((1.0 - np.cos(np.pi * t)) / 2.0) ** 2

def sched_cosine3(t):
    """Cosine³: ~90% time below μ/2"""
    return ((1.0 - np.cos(np.pi * t)) / 2.0) ** 3

def sched_tan(t):
    """Tan-based: ~90% below μ/2, rockets at end"""
    k = 0.95  # cutoff avoids singularity
    if t >= k:
        return 1.0
    return np.tan(np.pi * t / 2.0 * k) / np.tan(np.pi * k / 2.0)

def sched_sec2(t):
    """Sec²-based: very flat then explosive ramp"""
    k = 0.90
    if t >= k:
        return 1.0
    val  = (1.0 / np.cos(np.pi * t / 2.0 * k) ** 2 - 1.0)
    norm = (1.0 / np.cos(np.pi * k / 2.0) ** 2 - 1.0)
    return min(val / norm, 1.0)

def sched_delayed_cosine(t):
    """Delayed 50%: μ=0 for first half, cosine ramp in second half"""
    if t < 0.5:
        return 0.0
    t2 = (t - 0.5) / 0.5
    return (1.0 - np.cos(np.pi * t2)) / 2.0

def sched_delayed70_cosine(t):
    """Delayed 70%: μ=0 for 70%, cosine ramp in last 30%"""
    if t < 0.7:
        return 0.0
    t2 = (t - 0.7) / 0.3
    return (1.0 - np.cos(np.pi * t2)) / 2.0

def sched_cosec_inspired(t):
    """Cosec-inspired via 1 - 1/cosh(k*t): flat then fast"""
    k = 4.0
    val  = 1.0 - 1.0 / np.cosh(k * t)
    norm = 1.0 - 1.0 / np.cosh(k)
    return val / norm

# ── NEW: Power-law family (t^k) ───────────────────────────────────
# Simplest flattening — crosses μ/2 at t = 0.5^(1/k)
# No singularities, no cutoff, one parameter controls everything.

def sched_power5(t):
    """t^5:  87% broadcast"""
    return t ** 5

def sched_power10(t):
    """t^10: 93% broadcast"""
    return t ** 10

def sched_power20(t):
    """t^20: 97% broadcast"""
    return t ** 20

# ── NEW: Steep sigmoid (smoothed step) ───────────────────────────
# f(t) = sigmoid(k*(t - t0)), normalised to [0,1].
# k=30, t0=0.85 → essentially zero until ~0.7, smooth wall after.

def sched_sigmoid_step(t):
    """Sigmoid step: k=30 centred at t=0.85 — ~95% broadcast"""
    k, t0 = 30.0, 0.85
    def _sig(x): return 1.0 / (1.0 + np.exp(-x))
    lo = _sig(-k * t0)
    hi = _sig(k * (1.0 - t0))
    return (_sig(k * (t - t0)) - lo) / (hi - lo)

# ── NEW: Delayed linear (zero until d, then straight ramp) ────────

def sched_delayed80_linear(t):
    """Zero until t=0.8, linear ramp in last 20% — 90% broadcast"""
    d = 0.80
    if t < d:
        return 0.0
    return (t - d) / (1.0 - d)


schedules = {
    'linear':        sched_linear,
    'cosine':        sched_cosine,
    'cosine2':       sched_cosine2,
    'cosine3':       sched_cosine3,
    'tan':           sched_tan,
    'sec2':          sched_sec2,
    'delay50_cos':   sched_delayed_cosine,
    'delay70_cos':   sched_delayed70_cosine,
    'cosec_insp':    sched_cosec_inspired,
    'power5':        sched_power5,
    'power10':       sched_power10,
    'power20':       sched_power20,
    'sigmoid_step':  sched_sigmoid_step,
    'delay80_lin':   sched_delayed80_linear,
}

# ── Quick broadcast-time analysis (CPU, instant) ─────────────────
print('=' * 65)
print('SCHEDULE ANALYSIS: Time Spent Below μ/2')
print('=' * 65)
t_vals = np.linspace(0, 1, 1000)
for name, fn in schedules.items():
    vals = np.array([fn(t) for t in t_vals])
    b50 = (vals < 0.50).mean()
    b25 = (vals < 0.25).mean()
    b10 = (vals < 0.10).mean()
    print(f'  {name:>14}: below μ/2={b50:.0%}  below μ/4={b25:.0%}  below μ/10={b10:.0%}')


# ── Per-n comparison runner ───────────────────────────────────────

def run_schedule_comparison(n, num_instances=100, num_particles=2000,
                             alpha=3.0, mu_scale=0.1):
    steps = min(int(500 * np.sqrt(n)), 20000)
    engine = BSDTSonarEngine(
        n=n, num_instances=num_instances,
        num_particles=num_particles,
        alpha=alpha, mu_scale=mu_scale, device=device
    )
    cv, cs = engine.generate_instances()
    mu, lmax = engine.compute_mu(cv)

    print(f'\n{"="*65}')
    print(f'n = {n}  |  m = {int(alpha*n)}  |  steps = {steps}')
    print(f'μ_target = {mu.mean():.4f}  |  λ_max = {lmax.mean():.4f}')
    print(f'{"="*65}')

    results = {}
    for sched_name, sched_fn in schedules.items():
        if device.type == 'cuda':
            torch.cuda.empty_cache()

        def mu_override(step, total, mu_base, _fn=sched_fn):
            frac = step / max(total - 1, 1)
            return mu_base * _fn(frac)

        s_init = torch.clamp(torch.randn(engine.ni, num_particles, n, device=device) * 0.3, -0.9, 0.9)

        t0 = time.time()
        s_final, _ = engine.gradient_flow(
            s_init, cv, cs, mu, steps, dt=0.05, mu_override=mu_override
        )

        s_round = torch.sign(s_final + 1e-10)
        _, E_round, _ = engine.energy_and_grad(s_round, cv, cs, mu)
        best_E = E_round.min(dim=1).values
        solve_rate = (best_E < 0.5).float().mean().item()
        se   = np.sqrt(solve_rate * (1 - solve_rate) / num_instances)
        viol = best_E.clamp(min=0).mean().item()
        elapsed = time.time() - t0

        results[sched_name] = {'rate': solve_rate, 'se': se,
                                'viol': viol, 'time': elapsed}

        star = '★' if solve_rate >= 0.99 else '◆' if solve_rate >= 0.95 else ' '
        print(f'  {star} {sched_name:>14}: {solve_rate:6.1%} ± {se:.1%}  '
              f'viol={viol:.2f}  ({elapsed:.1f}s)')

    return results


# ── MAIN RUN ─────────────────────────────────────────────────────
torch.manual_seed(42)
np.random.seed(42)

print('\n' + '=' * 65)
print('H7 EXPERIMENT — ADVANCED ANNEALING SCHEDULES')
print('=' * 65)

all_results = {}
for n in [500, 750, 1000]:
    all_results[n] = run_schedule_comparison(n)


# ── SUMMARY TABLE ────────────────────────────────────────────────
print('\n' + '=' * 65)
print('H7 FULL SUMMARY')
print('=' * 65)
sched_names = list(schedules.keys())
hdr = f'{"Schedule":>14} | {"Bcast%":>6}'
for n in [500, 750, 1000]:
    hdr += f' | {"n="+str(n):>7}'
print(hdr)
print('-' * len(hdr))
for name in sched_names:
    vals  = np.array([schedules[name](t) for t in np.linspace(0, 1, 1000)])
    bcast = (vals < 0.5).mean()
    row   = f'{name:>14} | {bcast:>5.0%}'
    for n in [500, 750, 1000]:
        r = all_results.get(n, {}).get(name)
        row += f' | {r["rate"]:>6.1%}' if r else f' | {"—":>6}'
    print(row)


# ── CORRELATION ──────────────────────────────────────────────────
print('\n' + '=' * 65)
print('CORRELATION: Broadcast Time vs Solve Rate')
print('=' * 65)
for n in [500, 750, 1000]:
    bcasts = []
    rates  = []
    for name in sched_names:
        vals = np.array([schedules[name](t) for t in np.linspace(0, 1, 1000)])
        bcasts.append((vals < 0.5).mean())
        rates.append(all_results[n][name]['rate'])
    corr = np.corrcoef(bcasts, rates)[0, 1]
    tag  = 'POSITIVE → more broadcast = better' if corr > 0.5 else \
           'NEGATIVE → too much broadcast hurts' if corr < -0.5 else \
           'WEAK — ramp shape matters more than duration'
    print(f'  n={n}: r = {corr:+.3f}  ({tag})')


# ── VERDICT ──────────────────────────────────────────────────────
print('\n' + '=' * 65)
print('VERDICT: BEST SCHEDULE PER n')
print('=' * 65)
for n in [500, 750, 1000]:
    best  = max(all_results[n], key=lambda k: all_results[n][k]['rate'])
    brate = all_results[n][best]['rate']
    crate = all_results[n]['cosine']['rate']
    up    = brate - crate
    flag  = '★ SIGNIFICANT' if up > 0.05 else '◆ MODEST' if up > 0.01 else '= NEGLIGIBLE'
    print(f'  n={n}: best={best} ({brate:.1%})  cosine={crate:.1%}  uplift={up:+.1%}  [{flag}]')


# ── SCHEDULE PLOT ────────────────────────────────────────────────
t_arr = np.linspace(0, 1, 500)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
colors = plt.cm.tab10(np.linspace(0, 1, len(schedules)))
for (name, fn), col in zip(schedules.items(), colors):
    y = np.array([fn(t) for t in t_arr])
    ax.plot(t_arr, y, label=name, color=col, linewidth=2)
ax.axhline(0.5, color='k', linestyle='--', alpha=0.4, label='μ/2 threshold')
ax.set_xlabel('Annealing progress t')
ax.set_ylabel('μ(t) / μ_target')
ax.set_title('H7 Annealing Schedules')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

ax2 = axes[1]
ns  = [500, 750, 1000]
for (name, fn), col in zip(schedules.items(), colors):
    rates = [all_results[n][name]['rate'] * 100 for n in ns]
    ax2.plot(ns, rates, 'o-', label=name, color=col, linewidth=2, markersize=8)
ax2.axhline(93, linestyle='--', color='gray', alpha=0.5, label='cosine n=750 baseline (93%)')
ax2.set_xlabel('n (variables)')
ax2.set_ylabel('Solve rate (%)')
ax2.set_title('H7 Solve Rates vs n')
ax2.legend(fontsize=8)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('h7_results.png', dpi=150, bbox_inches='tight')
plt.show()
print('Plot saved: h7_results.png')